# NB-02 · Script de Inferencia Reproducible
## Proyecto GATOBYTE — Amazon Electronics Sentiment Analysis

**Requiere**: NB-01a, NB-01b y NB-01c ejecutados.

**Cambios respecto a la versión anterior**:
- `modo='tfidf'`: ahora usa `pipeline_transformacion_cpu.joblib` + `lightgbm_tuned_final_cpu.joblib`
  → **funciona sin GPU**, sin cuML, sin cuDF.
- `modo='stacking'`: usa exactamente los 3 modelos del proyecto
  (TF-IDF-CPU + Emb-LightGBM + DistilBERT-LoRA). El orden lo dicta `stacking_contract.json`.

**Contenido**:
- Clase `ClasificadorSentimiento` con 4 modos: `tfidf`, `embeddings`, `distilbert`, `stacking`
- Prueba con 10 reseñas nuevas (nunca vistas durante entrenamiento)
- Análisis comparativo de predicciones y tiempos
- Resumen ejecutivo del proyecto

## 0 · Entorno

In [ ]:
import os, sys, json, time, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
try:
    from google.colab import drive
    IN_COLAB = True
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/ML')
except ImportError:
    IN_COLAB = False
    BASE_DIR = Path('.')

DIRS = {
    'splits'                 : BASE_DIR / 'splits',
    'embeddings'             : BASE_DIR / 'embeddings',
    'models'                 : BASE_DIR / 'models',
    'models_preprocesamiento': BASE_DIR / 'models' / 'preprocesamiento',
    'models_lgbm'            : BASE_DIR / 'models' / 'lgbm_embeddings',
    'models_distilbert'      : BASE_DIR / 'models' / 'distilbert_lora',
    'models_stacking'        : BASE_DIR / 'models' / 'stacking',
    'models_cpu'             : BASE_DIR / 'models',          # pipeline_transformacion_cpu.joblib
    'migracion_cpu'          : BASE_DIR / 'models' / 'migracion_cpu',
    'reports'                : BASE_DIR / 'reports',
    'reports_comparacion'    : BASE_DIR / 'reports' / 'graficos_comparativos',
}

SEED = 42
PALETTE = {'positive':'#1D9E75','neutral':'#EF9F27','negative':'#E24B4A',
           'accent':'#378ADD','gray':'#888780','bg':'#F8F7F4'}
print('Entorno listo — BASE_DIR:', BASE_DIR)

Entorno listo — BASE_DIR: /content/drive/MyDrive/ML


In [ ]:
# ── Gestión de dependencias con restart automático ───────────────────────────
# torchao debe estar cargado ANTES que peft. Si la versión instalada es vieja,
# se actualiza y se reinicia el kernel una sola vez (flag en variable de entorno).

import subprocess, sys, importlib, importlib.util, os

def _ver(v):
    try: return tuple(int(x) for x in str(v).split('.')[:3])
    except: return (0, 0, 0)

_TORCHAO_MIN = (0, 16, 0)
_PEFT_MIN    = (0, 14, 0)
_FLAG        = 'NB02_TORCHAO_UPGRADED'   # evita bucle infinito de restarts

# ── Verificar torchao ─────────────────────────────────────────────────────────
_torchao_ok = False
if importlib.util.find_spec('torchao') is not None:
    try:
        import torchao as _tao
        _torchao_ok = _ver(_tao.__version__) >= _TORCHAO_MIN
    except Exception:
        pass

if not _torchao_ok:
    if os.environ.get(_FLAG) == '1':
        # Ya intentamos el upgrade y el kernel volvió a arrancar con versión vieja.
        # Colab tiene torchao pinneado — instalar en espacio de usuario.
        print('Instalando torchao en --user (entorno base tiene versión pinneada)...')
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', '-U',
            '--user', 'torchao>=0.16.0'
        ])
        # Forzar recarga desde espacio de usuario
        if importlib.util.find_spec('torchao') is not None:
            import importlib as _il
            import torchao as _tao_old
            _il.reload(_tao_old)
        print('✓ torchao instalado en --user. Continúa con la siguiente celda.')
    else:
        print('Actualizando torchao>=0.16.0 y reiniciando kernel...')
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', '-U', 'torchao>=0.16.0'
        ])
        os.environ[_FLAG] = '1'   # marcar antes del kill
        # Persistir el flag en un archivo (os.environ no sobrevive el restart)
        _flag_path = '/tmp/nb02_torchao_upgraded'
        open(_flag_path, 'w').write('1')
        print('✓ Instalado — reiniciando kernel (normal en Colab)...')
        os.kill(os.getpid(), 9)
else:
    import torchao as _tao
    print(f'✓ torchao {_tao.__version__}')
    # Limpiar flag si existe
    try: os.remove('/tmp/nb02_torchao_upgraded')
    except: pass

# Leer flag desde archivo (sobrevive el restart, a diferencia de os.environ)
if os.path.exists('/tmp/nb02_torchao_upgraded'):
    os.environ[_FLAG] = '1'

# ── Verificar / instalar peft ─────────────────────────────────────────────────
_peft_ok = False
if importlib.util.find_spec('peft') is not None:
    try:
        import peft as _peft
        _peft_ok = _ver(_peft.__version__) >= _PEFT_MIN
    except Exception:
        pass

if not _peft_ok:
    print('Instalando peft>=0.14.0 + transformers + accelerate...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'peft>=0.14.0', 'transformers>=4.40.0', 'accelerate>=0.30.0'
    ])
    print('✓ peft instalado')
else:
    import peft as _peft
    print(f'✓ peft {_peft.__version__}')

print('✓ Dependencias listas')

✓ torchao 0.17.0
✓ peft 0.19.1
✓ Dependencias listas


## 1 · Importar clases CPU del pipeline migrado

In [ ]:
# El pipeline CPU se importa desde el script de migración.
# Esto evita duplicar las clases y garantiza compatibilidad con el joblib exportado.
# Asegúrate de que migrar_pipeline_cpu.py esté en BASE_DIR/migracion_cpu/ o en el PATH.

import importlib.util, sys

# Intentar importar desde la carpeta migracion_cpu relativa a BASE_DIR
_ruta_script = BASE_DIR / 'models' / 'migracion_cpu' / 'migrar_pipeline_cpu.py'

if _ruta_script.exists():
    spec = importlib.util.spec_from_file_location('migrar_pipeline_cpu', str(_ruta_script))
    _mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(_mod)
    # Registrar las clases en __main__ para que joblib las encuentre al deserializar
    import __main__
    __main__.CpuFullPreprocessor = _mod.CpuFullPreprocessor
    __main__.CpuPreprocessor     = _mod.CpuPreprocessor
    __main__.CpuTextCleaner      = _mod.CpuTextCleaner
    print(f'✓ Clases CPU cargadas desde: {_ruta_script}')
else:
    # Fallback: definir inline si el script no está disponible
    print(f'Aviso: {_ruta_script} no encontrado — definiendo clases CPU inline')
    import re
    import scipy.sparse as sp
    from sklearn.base import BaseEstimator, TransformerMixin
    from sklearn.preprocessing import StandardScaler
    from sklearn.feature_extraction.text import TfidfVectorizer

    class CpuPreprocessor(BaseEstimator, TransformerMixin):
        def __init__(self, p1_price, p99_price, p1_len, p99_len, cat_cols):
            self.p1_price=p1_price; self.p99_price=p99_price
            self.p1_len=p1_len; self.p99_len=p99_len; self.cat_cols=cat_cols
        def fit(self, X, y=None): return self
        def transform(self, X):
            import pandas as pd
            X = X.copy()
            X['title'] = X['title'].fillna('')
            X['text']  = X['text'].fillna('')
            X['main_category'] = X['main_category'].fillna('Unknown')
            X['text_combined'] = X['title'].str.strip() + ' ' + X['text'].str.strip()
            X['price']    = X['price'].clip(self.p1_price, self.p99_price)
            X['text_len'] = X['text_len'].clip(self.p1_len, self.p99_len)
            dummies = pd.get_dummies(X['main_category'], prefix='cat')
            for col in [f'cat_{c}' for c in self.cat_cols]:
                if col not in dummies.columns: dummies[col] = 0
            dummies = dummies[[f'cat_{c}' for c in self.cat_cols]].astype('float32')
            return pd.concat([X[['text_len','price']], dummies, X[['text_combined']]], axis=1)

    class CpuTextCleaner(BaseEstimator, TransformerMixin):
        _ACCENT_MAP = str.maketrans('áéíóúüñÁÉÍÓÚÜÑ','aeiouunAEIOUUN')
        def fit(self, X, y=None): return self
        def transform(self, X):
            s = X.fillna('').str.lower()
            s = s.str.translate(self._ACCENT_MAP)
            s = s.str.replace(r'[^a-z\s]',' ',regex=True)
            s = s.str.replace(r'\s+',' ',regex=True)
            return s.str.strip()

    class CpuFullPreprocessor(BaseEstimator, TransformerMixin):
        def __init__(self, params):
            self.params=params; self._build_inner()
        def _build_inner(self):
            p=self.params
            self._prep=CpuPreprocessor(**{k:p['clip'][k] for k in ['p1_price','p99_price','p1_len','p99_len','cat_cols']})
            sc_p=p['scaler']
            self._scaler=StandardScaler(with_mean=sc_p['with_mean'],with_std=sc_p['with_std'])
            self._scaler.mean_=np.array(sc_p['mean_']); self._scaler.var_=np.array(sc_p['var_'])
            self._scaler.scale_=np.array(sc_p['scale_']); self._scaler.n_features_in_=sc_p['n_features_in_']
            self._scaler.n_samples_seen_=sc_p['n_samples_seen_']
            self._scaler.feature_names_in_=np.array(['text_len','price'])
            tf_p=p['tfidf']
            self._tfidf=TfidfVectorizer(vocabulary=tf_p['vocabulary'],analyzer=tf_p['analyzer'],
                lowercase=tf_p['lowercase'],max_df=tf_p['max_df'],min_df=tf_p['min_df'],
                max_features=tf_p['max_features'],ngram_range=tuple(tf_p['ngram_range']),
                binary=tf_p['binary'],norm=tf_p['norm'],use_idf=tf_p['use_idf'],
                smooth_idf=tf_p['smooth_idf'],sublinear_tf=tf_p['sublinear_tf'])
            dummy_text=' '.join(list(tf_p['vocabulary'].keys())[:50])
            self._tfidf.fit([dummy_text])
            self._tfidf._tfidf.idf_=np.array(tf_p['idf_values'],dtype=np.float64)
            self._cleaner=CpuTextCleaner()
        def fit(self, X, y=None): return self
        def __sklearn_is_fitted__(self): return True
        def transform(self, X):
            prep_df=self._prep.transform(X)
            num_scaled=self._scaler.transform(prep_df[['text_len','price']])
            num_sparse=sp.csr_matrix(num_scaled.astype(np.float32))
            cat_cols_in=[c for c in prep_df.columns if c.startswith('cat_')]
            cat_sparse=sp.csr_matrix(prep_df[cat_cols_in].values.astype(np.float32))
            cleaned=self._cleaner.transform(prep_df['text_combined'])
            tfidf_sparse=self._tfidf.transform(cleaned)
            return sp.hstack([num_sparse,cat_sparse,tfidf_sparse],format='csr')
        def fit_transform(self, X, y=None): return self.transform(X)
        @property
        def cat_cols_(self): return self._prep.cat_cols

    import __main__
    __main__.CpuFullPreprocessor=CpuFullPreprocessor
    __main__.CpuPreprocessor=CpuPreprocessor
    __main__.CpuTextCleaner=CpuTextCleaner
    print('✓ Clases CPU definidas inline')

✓ Clases CPU cargadas desde: /content/drive/MyDrive/ML/models/migracion_cpu/migrar_pipeline_cpu.py


## 2 · Clase `ClasificadorSentimiento`

In [ ]:
class ClasificadorSentimiento:
    """
    Clasificador de sentimiento para reseñas de Amazon Electronics.

    Modos disponibles
    -----------------
    'tfidf'      → Pipeline CPU (pipeline_transformacion_cpu.joblib) + LightGBM CPU.
                   Funciona completamente sin GPU. Mismas métricas que el original GPU.
    'embeddings' → LightGBM sobre embeddings MiniLM-L6-v2. CPU siempre.
    'distilbert' → DistilBERT + LoRA. CPU (más lento) o GPU si disponible.
    'stacking'   → Meta-learner LogReg sobre 3 modelos: TF-IDF-CPU + Emb-LightGBM + DistilBERT.
                   Orden de concatenación definido por stacking_contract.json.

    Entrada (predecir)
    ------------------
    Lista de dicts con campos:
      - text (str)              ← OBLIGATORIO
      - title (str)             ← recomendado
      - price (float)           ← opcional, default=0
      - text_len (int)          ← opcional, se calcula si falta
      - main_category (str)     ← recomendado, default='Electronics'
    """

    MODOS = ('tfidf', 'embeddings', 'distilbert', 'stacking')
    FEATURES_NUMERIC = ['text_len', 'price']
    FEATURES_CAT     = ['main_category']

    def __init__(self, modo: str = 'stacking', base_dir: Path = None):
        assert modo in self.MODOS, f'modo debe ser uno de {self.MODOS}'
        self.modo  = modo
        self.base  = Path(base_dir) if base_dir else BASE_DIR
        self._dirs = {
            'models'             : self.base / 'models',
            'models_preprocesamiento': self.base / 'models' / 'preprocesamiento',
            'models_lgbm'        : self.base / 'models' / 'lgbm_embeddings',
            'models_distilbert'  : self.base / 'models' / 'distilbert_lora',
            'models_stacking'    : self.base / 'models' / 'stacking',
            # Pipeline CPU en la carpeta que queda un nivel arriba de migracion_cpu
            'models_cpu'         : self.base / 'models',
        }
        self.le = joblib.load(self._dirs['models'] / 'label_encoder.joblib')
        self._cargar(modo)

    def _cargar(self, modo):
        import torch
        self._gpu    = torch.cuda.is_available()
        self._device = 'cuda' if self._gpu else 'cpu'

        if modo in ('tfidf', 'stacking'):
            # Pipeline CPU — compatible con cualquier entorno, sin cuML
            _pipe_cpu_path  = self._dirs['models_cpu'] / 'pipeline_transformacion_cpu.joblib'
            _lgbm_cpu_path  = self._dirs['models_cpu'] / 'lightgbm_tuned_final_cpu.joblib'
            try:
                self._pipe_tfidf = joblib.load(_pipe_cpu_path)
                self._lgbm_tfidf = joblib.load(_lgbm_cpu_path)
                self._tfidf_disponible = True
                print(f'  ✓ Pipeline TF-IDF CPU cargado desde {_pipe_cpu_path.name}')
            except Exception as _e:
                self._tfidf_disponible = False
                print(f'  Aviso: pipeline TF-IDF CPU no disponible ({type(_e).__name__}: {_e})')
                print(f'         Ruta esperada: {_pipe_cpu_path}')
                if modo == 'tfidf':
                    raise RuntimeError(
                        f'Modo tfidf requiere {_pipe_cpu_path.name} y {_lgbm_cpu_path.name}. '
                        'Ejecuta models/migracion_cpu/migrar_pipeline_cpu.py primero.'
                    ) from _e

        if modo in ('embeddings', 'stacking'):
            from sentence_transformers import SentenceTransformer
            self._st = SentenceTransformer(
                'sentence-transformers/all-MiniLM-L6-v2', device=self._device)
            self._st.max_seq_length = 128
            self._st.eval()
            for p in self._st.parameters(): p.requires_grad = False

            self._pipe_emb = joblib.load(self._dirs['models_lgbm'] / 'emb_lgbm_pipeline.joblib')

            self._tab_prep = joblib.load(
                self._dirs['models_preprocesamiento'] / 'emb_tabular_preprocessor.joblib')
            clip_params = joblib.load(
                self._dirs['models_preprocesamiento'] / 'emb_clip_params.joblib')
            self._clip_p1  = clip_params['p1']
            self._clip_p99 = clip_params['p99']

            _contract_path = self._dirs['models_preprocesamiento'] / 'emb_vector_contract.json'
            if _contract_path.exists():
                with open(_contract_path) as _f:
                    self._vector_contract = json.load(_f)
                print(f'  ✓ Contrato del vector: {self._vector_contract["total_dims"]} dims')
            else:
                self._vector_contract = None

        if modo in ('distilbert', 'stacking'):
            import torch
            from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
            from peft import PeftModel
            lora_path = self._dirs['models_distilbert']
            self._tok_db = DistilBertTokenizerFast.from_pretrained(str(lora_path))
            base = DistilBertForSequenceClassification.from_pretrained(
                'distilbert-base-uncased', num_labels=3, ignore_mismatched_sizes=True)
            self._db = PeftModel.from_pretrained(base, str(lora_path)).to(self._device)
            self._db.eval()
            self._torch = torch
            print(f'  ✓ DistilBERT+LoRA cargado (device={self._device})')

        if modo == 'stacking':
            self._meta = joblib.load(self._dirs['models_stacking'] / 'stacking_meta_pipeline.joblib')
            # Leer contrato de stacking para orden canónico de concatenación
            _stk_contract = self._dirs['models_stacking'] / 'stacking_contract.json'
            if _stk_contract.exists():
                with open(_stk_contract) as _f:
                    self._stacking_contract = json.load(_f)
            else:
                self._stacking_contract = {'modelos_base_orden': ['Emb-LightGBM', 'DistilBERT-LoRA']}
            print(f'  ✓ Stacking meta-learner cargado. Orden: {self._stacking_contract["modelos_base_orden"]}')

        print(f'✓ ClasificadorSentimiento cargado — modo={modo}  device={self._device}')

    def _normalizar(self, resenas):
        out = []
        for r in resenas:
            assert 'text' in r and str(r['text']).strip(), "Cada reseña debe tener 'text' no vacío"
            n = dict(
                title         = str(r.get('title', '')),
                text          = str(r['text']),
                price         = float(r.get('price', 0.0)),
                main_category = str(r.get('main_category', 'Electronics')),
            )
            n['text_len'] = int(r.get('text_len', len(n['text'])))
            out.append(n)
        return out

    def _texto_completo(self, resenas):
        return [(r['title'] + ' ' + r['text']).strip() for r in resenas]

    def _tab_features(self, resenas):
        df_in = pd.DataFrame({
            'text_len'     : [float(r['text_len']) for r in resenas],
            'price'        : [float(r['price'])    for r in resenas],
            'main_category': [str(r['main_category']) for r in resenas],
        })
        for col in self.FEATURES_NUMERIC:
            df_in[col] = df_in[col].clip(self._clip_p1[col], self._clip_p99[col])
        tab = self._tab_prep.transform(
            df_in[self.FEATURES_NUMERIC + self.FEATURES_CAT]).astype(np.float32)
        if self._vector_contract is not None:
            expected = self._vector_contract['tab_dims']
            assert tab.shape[1] == expected, f'Dim tabular incorrecta: esperada {expected}, obtenida {tab.shape[1]}'
        return tab

    def _probs_tfidf(self, resenas):
        """Pipeline CPU — no requiere GPU ni cuML."""
        if not getattr(self, '_tfidf_disponible', False):
            raise RuntimeError('Pipeline TF-IDF CPU no disponible. '
                               'Ejecuta models/migracion_cpu/migrar_pipeline_cpu.py primero.')
        df_in = pd.DataFrame(resenas)
        X = self._pipe_tfidf.transform(df_in)
        return self._lgbm_tfidf.predict_proba(X)

    def _probs_embeddings(self, resenas):
        textos = self._texto_completo(resenas)
        emb = self._st.encode(textos, batch_size=256,
                              convert_to_numpy=True, normalize_embeddings=True)
        tab = self._tab_features(resenas)
        X   = np.hstack([emb, tab]).astype(np.float32)
        if self._vector_contract is not None:
            expected_dims = self._vector_contract['total_dims']
            assert X.shape[1] == expected_dims, f'Shape incorrecto: esperado (N,{expected_dims}), obtenido {X.shape}'
        return self._pipe_emb.predict_proba(X)

    def _probs_distilbert(self, resenas):
        textos = self._texto_completo(resenas)
        enc = self._tok_db(textos, truncation=True, padding=True,
                           max_length=128, return_tensors='pt')
        enc = {k: v.to(self._device) for k, v in enc.items()}
        with self._torch.no_grad():
            logits = self._db(**enc).logits
        return self._torch.softmax(logits, -1).cpu().numpy()

    def predecir(self, resenas: list) -> list:
        rns = self._normalizar(resenas)

        if self.modo == 'tfidf':
            proba = self._probs_tfidf(rns)

        elif self.modo == 'embeddings':
            proba = self._probs_embeddings(rns)

        elif self.modo == 'distilbert':
            proba = self._probs_distilbert(rns)

        elif self.modo == 'stacking':
            # Construir meta-features en el orden EXACTO del stacking_contract.json
            _orden = self._stacking_contract['modelos_base_orden']
            _probs_map = {}

            # Intentar cada modelo base según el contrato
            if 'TF-IDF-LightGBM' in _orden:
                try:
                    _probs_map['TF-IDF-LightGBM'] = self._probs_tfidf(rns)
                except Exception as _e:
                    print(f'  TF-IDF no disponible: {_e}')

            if 'Emb-LightGBM' in _orden:
                try:
                    _probs_map['Emb-LightGBM'] = self._probs_embeddings(rns)
                except Exception as _e:
                    print(f'  Embeddings no disponible: {_e}')

            if 'DistilBERT-LoRA' in _orden:
                try:
                    _probs_map['DistilBERT-LoRA'] = self._probs_distilbert(rns)
                except Exception as _e:
                    print(f'  DistilBERT no disponible: {_e}')

            # Concatenar en el orden del contrato (solo los disponibles)
            probas_disponibles = [_probs_map[m] for m in _orden if m in _probs_map]
            assert len(probas_disponibles) >= 2, (
                f'Stacking necesita ≥2 modelos base. '
                f'Disponibles: {list(_probs_map.keys())}. '
                f'Requeridos: {_orden}'
            )
            if len(probas_disponibles) < len(_orden):
                faltantes = [m for m in _orden if m not in _probs_map]
                print(f'  Aviso: {faltantes} no disponibles — stacking con {len(probas_disponibles)} modelos')

            meta  = np.hstack(probas_disponibles)
            proba = self._meta.predict_proba(meta)

        y_pred = proba.argmax(axis=1)
        clases = self.le.inverse_transform(y_pred)

        return [
            {
                'sentimiento'   : c,
                'confianza'     : float(proba[i].max()),
                'probabilidades': {cls: float(proba[i, j])
                                   for j, cls in enumerate(self.le.classes_)},
            }
            for i, c in enumerate(clases)
        ]

print('✓ ClasificadorSentimiento definido')

✓ ClasificadorSentimiento definido


## 3 · Nuevas reseñas de prueba

In [ ]:
NUEVAS_RESENAS = [
    {'title': 'Best wireless headphones I have owned',
     'text' : 'Crystal clear sound, incredibly comfortable even after 5 hours. '
              'Battery lasts exactly 30 hours as advertised. Zero connectivity issues.',
     'price': 89.99, 'main_category': 'Electronics'},
    {'title': 'Exceeded all my expectations',
     'text' : 'Fast delivery, premium packaging, product quality is outstanding. '
              'Highly recommend to anyone looking for a reliable electronic device.',
     'price': 55.00, 'main_category': 'Electronics'},
    {'title': 'Complete waste of money — DO NOT BUY',
     'text' : 'Stopped working after exactly 3 days. Plastic feels cheap and brittle. '
              'Customer service was completely useless. Returned immediately.',
     'price': 34.99, 'main_category': 'Electronics'},
    {'title': 'Broken on arrival',
     'text' : 'The device was damaged when I opened the box. Screen had dead pixels '
              'and the charging port was loose. Total disappointment.',
     'price': 120.00, 'main_category': 'Computers'},
    {'title': 'Decent product but nothing special',
     'text' : 'Works as described. Build quality is average. Packaging was slightly '
              'damaged but product was fine. Does the job, nothing more.',
     'price': 22.00, 'main_category': 'Electronics'},
    {'title': 'Mixed feelings about this one',
     'text' : 'Sound quality is good but the battery only lasts 12 hours instead of '
              'the advertised 20. Comfortable to wear but the price feels a bit high.',
     'price': 75.00, 'main_category': 'Electronics'},
    {'title': 'Great product if you enjoy throwing money away',
     'text' : 'Oh sure, it works perfectly — for exactly one week. Then the magic '
              'smoke escaped and now it is a very expensive paperweight.',
     'price': 89.99, 'main_category': 'Electronics'},
    {'title': 'Perfect',
     'text' : 'Love it. Fast shipping.',
     'price': 15.00, 'main_category': 'Electronics'},
    {'title': 'Detailed review after 6 months of daily use',
     'text' : 'After six months I can give a thorough review. The processor handles '
              'multitasking well and the display colors are accurate for photo editing. '
              'However, the fan noise under load is significant and the battery degrades '
              'noticeably after 200 charge cycles. Overall a capable machine with caveats.',
     'price': 850.00, 'main_category': 'Computers'},
    {'title': '',
     'text' : 'Not bad for the price. I have seen better products but this one gets '
              'the job done. Would not buy again but would not return it either.',
     'price': 28.00, 'main_category': 'Electronics'},
]

ETIQUETA_ESPERADA = [
    'positive', 'positive',
    'negative', 'negative',
    'neutral',  'neutral',
    'negative',  # ironía
    'positive',
    'neutral',
    'neutral',
]

print(f'Nuevas reseñas preparadas: {len(NUEVAS_RESENAS)}')
for i, (r, e) in enumerate(zip(NUEVAS_RESENAS, ETIQUETA_ESPERADA)):
    print(f'  [{i+1}] Esperado: {e:<10}  Título: {r["title"][:50] or "(sin título)"}')

Nuevas reseñas preparadas: 10
  [1] Esperado: positive    Título: Best wireless headphones I have owned
  [2] Esperado: positive    Título: Exceeded all my expectations
  [3] Esperado: negative    Título: Complete waste of money — DO NOT BUY
  [4] Esperado: negative    Título: Broken on arrival
  [5] Esperado: neutral     Título: Decent product but nothing special
  [6] Esperado: neutral     Título: Mixed feelings about this one
  [7] Esperado: negative    Título: Great product if you enjoy throwing money away
  [8] Esperado: positive    Título: Perfect
  [9] Esperado: neutral     Título: Detailed review after 6 months of daily use
  [10] Esperado: neutral     Título: (sin título)


## 4 · Ejecutar inferencia con todos los modos

In [ ]:
resultados_por_modo = {}
tiempos_inferencia  = {}

# Todos los modos disponibles — tfidf ahora funciona en CPU
MODOS_PROBAR = ['tfidf', 'embeddings', 'distilbert', 'stacking']

print('\n' + '='*70)
print('  INFERENCIA CON NUEVAS RESEÑAS')
print('='*70)

for modo in MODOS_PROBAR:
    try:
        clf = ClasificadorSentimiento(modo=modo)
        t0  = time.time()
        resultados = clf.predecir(NUEVAS_RESENAS)
        t_inf = time.time() - t0

        resultados_por_modo[modo] = resultados
        tiempos_inferencia[modo]  = round(t_inf, 3)

        print(f'\n  Modo: {modo.upper()}  ({t_inf*1000:.1f} ms para {len(NUEVAS_RESENAS)} reseñas)')
        print(f'  {"─"*65}')
        correctos = 0
        for i, (res, esperado) in enumerate(zip(resultados, ETIQUETA_ESPERADA)):
            emoji = {'positive':'🟢','neutral':'🟡','negative':'🔴'}[res['sentimiento']]
            ok    = '✓' if res['sentimiento'] == esperado else '✗'
            if res['sentimiento'] == esperado: correctos += 1
            print(f'  {ok} {emoji} [{res["sentimiento"]:<10}] conf={res["confianza"]:.1%}  '
                  f'Esperado: {esperado:<10}  {NUEVAS_RESENAS[i]["title"][:35] or "(sin título)"}')
        print(f'  Exactitud en nuevos datos: {correctos}/{len(NUEVAS_RESENAS)} '
              f'({correctos/len(NUEVAS_RESENAS)*100:.0f}%)')

    except Exception as e:
        print(f'\n  ⚠ Modo {modo}: {type(e).__name__}: {str(e)[:100]}')


  INFERENCIA CON NUEVAS RESEÑAS
  ✓ Pipeline TF-IDF CPU cargado desde pipeline_transformacion_cpu.joblib
✓ ClasificadorSentimiento cargado — modo=tfidf  device=cpu

  Modo: TFIDF  (311.1 ms para 10 reseñas)
  ─────────────────────────────────────────────────────────────────
  ✓ 🟢 [positive  ] conf=88.6%  Esperado: positive    Best wireless headphones I have own
  ✓ 🟢 [positive  ] conf=96.2%  Esperado: positive    Exceeded all my expectations
  ✓ 🔴 [negative  ] conf=99.2%  Esperado: negative    Complete waste of money — DO NOT BU
  ✓ 🔴 [negative  ] conf=89.3%  Esperado: negative    Broken on arrival
  ✓ 🟡 [neutral   ] conf=93.0%  Esperado: neutral     Decent product but nothing special
  ✓ 🟡 [neutral   ] conf=78.3%  Esperado: neutral     Mixed feelings about this one
  ✗ 🟢 [positive  ] conf=83.2%  Esperado: negative    Great product if you enjoy throwing
  ✓ 🟢 [positive  ] conf=98.5%  Esperado: positive    Perfect
  ✓ 🟡 [neutral   ] conf=53.3%  Esperado: neutral     Detailed review aft

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  ✓ Contrato del vector: 422 dims


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ✓ DistilBERT+LoRA cargado (device=cpu)
  ✓ Stacking meta-learner cargado. Orden: ['TF-IDF-LightGBM', 'Emb-LightGBM', 'DistilBERT-LoRA']
✓ ClasificadorSentimiento cargado — modo=stacking  device=cpu

  Modo: STACKING  (1379.2 ms para 10 reseñas)
  ─────────────────────────────────────────────────────────────────
  ✓ 🟢 [positive  ] conf=98.7%  Esperado: positive    Best wireless headphones I have own
  ✓ 🟢 [positive  ] conf=99.1%  Esperado: positive    Exceeded all my expectations
  ✓ 🔴 [negative  ] conf=98.5%  Esperado: negative    Complete waste of money — DO NOT BU
  ✓ 🔴 [negative  ] conf=96.9%  Esperado: negative    Broken on arrival
  ✓ 🟡 [neutral   ] conf=97.6%  Esperado: neutral     Decent product but nothing special
  ✓ 🟡 [neutral   ] conf=94.1%  Esperado: neutral     Mixed feelings about this one
  ✗ 🟢 [positive  ] conf=94.0%  Esperado: negative    Great product if you enjoy throwing
  ✓ 🟢 [positive  ] conf=99.2%  Esperado: positive    Perfect
  ✓ 🟡 [neutral   ] conf=51.6%  Es

## 5 · Análisis comparativo

In [ ]:
if len(resultados_por_modo) >= 2:
    print('\nComparación de predicciones por reseña:')
    header = f'  {"Título":<40}  ' + '  '.join(f'{m[:10]:<11}' for m in resultados_por_modo)
    print(header)
    print('  ' + '─'*90)
    for i, r in enumerate(NUEVAS_RESENAS):
        preds = []
        for modo_k in resultados_por_modo:
            res = resultados_por_modo[modo_k][i]
            preds.append(f"{res['sentimiento'][:3]}({res['confianza']:.0%})")
        titulo = r['title'][:40] or '(sin título)'
        print(f"  {titulo:<40}  {'  '.join(p.ljust(11) for p in preds)}")

    print(f'\nTiempos de inferencia ({len(NUEVAS_RESENAS)} reseñas):')
    for modo_k, t in tiempos_inferencia.items():
        ms = t * 1000
        print(f'  {modo_k:<14}: {ms:.1f} ms total  ({ms/len(NUEVAS_RESENAS):.1f} ms/reseña)')


Comparación de predicciones por reseña:
  Título                                    tfidf        embeddings   distilbert   stacking   
  ──────────────────────────────────────────────────────────────────────────────────────────
  Best wireless headphones I have owned     pos(89%)     pos(89%)     pos(100%)    pos(99%)   
  Exceeded all my expectations              pos(96%)     pos(99%)     pos(100%)    pos(99%)   
  Complete waste of money — DO NOT BUY      neg(99%)     neg(99%)     neg(100%)    neg(99%)   
  Broken on arrival                         neg(89%)     neg(94%)     neg(96%)     neg(97%)   
  Decent product but nothing special        neu(93%)     neu(51%)     neu(80%)     neu(98%)   
  Mixed feelings about this one             neu(78%)     neu(62%)     neu(77%)     neu(94%)   
  Great product if you enjoy throwing mone  pos(83%)     neg(70%)     pos(83%)     pos(94%)   
  Perfect                                   pos(99%)     pos(100%)    pos(100%)    pos(99%)   
  Detailed 

## 6 · Guardar resultados

In [ ]:
from pathlib import Path

_reports_dir = BASE_DIR / 'reports'
_reports_dir.mkdir(parents=True, exist_ok=True)

inferencia_output = {
    'timestamp'       : datetime.now().isoformat(),
    'n_resenas'       : len(NUEVAS_RESENAS),
    'modos_ejecutados': list(resultados_por_modo.keys()),
    'tiempos_ms'      : {k: round(v*1000, 2) for k, v in tiempos_inferencia.items()},
    'resultados'      : {
        modo: [
            {
                'resena_idx'    : i,
                'titulo'        : NUEVAS_RESENAS[i]['title'],
                'esperado'      : ETIQUETA_ESPERADA[i],
                'predicho'      : res['sentimiento'],
                'correcto'      : res['sentimiento'] == ETIQUETA_ESPERADA[i],
                'confianza'     : res['confianza'],
                'probabilidades': res['probabilidades'],
            }
            for i, res in enumerate(resultados_por_modo[modo])
        ]
        for modo in resultados_por_modo
    },
}

DEMO_PATH = _reports_dir / 'demo_inferencia_nuevos_datos.json'
with open(DEMO_PATH, 'w', encoding='utf-8') as f:
    json.dump(inferencia_output, f, indent=2, ensure_ascii=False)

print(f'✓ Resultados guardados: {DEMO_PATH.name}')

✓ Resultados guardados: demo_inferencia_nuevos_datos.json


## 7 · Resumen ejecutivo del proyecto

In [ ]:
CSV_PATH = BASE_DIR / 'reports' / 'graficos_comparativos' / 'tabla_maestra_comparacion.csv'
if CSV_PATH.exists():
    df_comp = pd.read_csv(CSV_PATH, index_col=0)

    print('\n' + '='*70)
    print('  RESUMEN EJECUTIVO — PROYECTO GATOBYTE')
    print('='*70)
    print(f'  Dataset    : Amazon Electronics Reviews — 1M filas')
    print(f'  Splits     : Train 70% | Val 15% | Test 15% (estratificado, SEED=42)')
    print(f'  Transformer: all-MiniLM-L6-v2 (inferencia pura) + DistilBERT+LoRA (0.44% params)')
    print(f'  Pipeline   : TF-IDF migrado a CPU (pipeline_transformacion_cpu.joblib)')
    print(f'  Stacking   : 3 modelos (TF-IDF-CPU + Emb-LightGBM + DistilBERT-LoRA)')

    if 'Score-Compuesto' in df_comp.columns:
        score_col = 'Score-Compuesto'
    elif 'score_compuesto' in df_comp.columns:
        score_col = 'score_compuesto'
    else:
        score_col = df_comp.columns[0]

    ganador = df_comp[score_col].idxmax()
    print(f'\n  MODELO GANADOR: {ganador}')

    cols_display = {
        'f1_macro': 'F1-Macro', 'F1-Macro': 'F1-Macro',
        'balanced_accuracy': 'Balanced-Acc', 'Balanced-Acc': 'Balanced-Acc',
        'roc_auc_macro': 'ROC-AUC', 'ROC-AUC': 'ROC-AUC',
    }
    for col_cand, label in cols_display.items():
        if col_cand in df_comp.columns:
            print(f'    {label:<18}: {df_comp.loc[ganador, col_cand]:.4f}')
    print(f'    Score Compuesto  : {df_comp.loc[ganador, score_col]:.4f}')

    f1_col  = 'f1_macro'  if 'f1_macro'  in df_comp.columns else 'F1-Macro'
    acc_col = 'accuracy'  if 'accuracy'  in df_comp.columns else 'Accuracy'
    if f1_col in df_comp.columns:
        f1_g = df_comp.loc[ganador, f1_col]
        print(f'\n  Criterios de éxito:')
        print(f'    F1-Macro ≥ 0.65: {"✓" if f1_g >= 0.65 else "✗"}  ({f1_g:.4f})')
    if acc_col in df_comp.columns:
        acc_g = df_comp.loc[ganador, acc_col]
        print(f'    Accuracy ≥ 0.80: {"✓" if acc_g >= 0.80 else "✗"}  ({acc_g:.4f})')

    print('\n  Política Transformers:')
    print('    all-MiniLM-L6-v2: inferencia pura, 0 parámetros entrenados ✓')
    print('    DistilBERT+LoRA : 0.44% parámetros, ajuste muy liviano ✓')
    print('    TF-IDF pipeline : migrado a CPU, funciona sin GPU ✓')
    print('='*70)
else:
    print('Tabla maestra no encontrada — ejecuta NB-01c primero')


  RESUMEN EJECUTIVO — PROYECTO GATOBYTE
  Dataset    : Amazon Electronics Reviews — 1M filas
  Splits     : Train 70% | Val 15% | Test 15% (estratificado, SEED=42)
  Transformer: all-MiniLM-L6-v2 (inferencia pura) + DistilBERT+LoRA (0.44% params)
  Pipeline   : TF-IDF migrado a CPU (pipeline_transformacion_cpu.joblib)
  Stacking   : 3 modelos (TF-IDF-CPU + Emb-LightGBM + DistilBERT-LoRA)

  MODELO GANADOR: DistilBERT+LoRA
    F1-Macro          : 0.7272
    Balanced-Acc      : 0.7833
    ROC-AUC           : 0.9483
    Score Compuesto  : 0.8104

  Criterios de éxito:
    F1-Macro ≥ 0.65: ✓  (0.7272)
    Accuracy ≥ 0.80: ✓  (0.8511)

  Política Transformers:
    all-MiniLM-L6-v2: inferencia pura, 0 parámetros entrenados ✓
    DistilBERT+LoRA : 0.44% parámetros, ajuste muy liviano ✓
    TF-IDF pipeline : migrado a CPU, funciona sin GPU ✓
